In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

In [ ]:
from jetnet.datasets import JetNet
from jetnet.datasets.normalisations import FeaturewiseLinearBounded, FeaturewiseLinear

In [ ]:
MASK = True
NUM_PARTICLES = 30
TRAIN_SPLIT = 0.7

data_args = {
    "jet_type": ["g", "q", "t"],
    "data_dir": "datasets/jetnet",
    "num_particles": NUM_PARTICLES,
    "particle_features": (
        JetNet.ALL_PARTICLE_FEATURES if MASK else JetNet.ALL_PARTICLE_FEATURES[:-1]
    ),
    # The order of the list is preserved in the retrieved data
    "jet_features": ["eta", "pt", "mass", "num_particles", "type"],
    # "particle_normalisation": particle_normalizer,
    "split_fraction": [TRAIN_SPLIT, 1 - TRAIN_SPLIT, 0],
    "download": True
}

In [ ]:
from util.coordinates import transform_rel_particle_coordinates_to_cartesian
X_train = JetNet(**data_args, split="train")
X_test = JetNet(**data_args, split="valid")
X_train_particle_transformed = transform_rel_particle_coordinates_to_cartesian(X_train)
X_test_particle_transformed = transform_rel_particle_coordinates_to_cartesian(X_test)

print(X_train_particle_transformed.shape)
print(X_test_particle_transformed.shape)

In [ ]:
e_c = np.array(X_train_particle_transformed[:, :, 0].flatten())
p_x = np.array(X_train_particle_transformed[:, :, 1].flatten())
p_y = np.array(X_train_particle_transformed[:, :, 2].flatten())
p_z = np.array(X_train_particle_transformed[:, :, 3].flatten())

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 10))
axs[0, 0].hist(e_c, bins=100, alpha=0.5, color='blue')
axs[0, 0].set_title(r'$E/c$')
axs[0, 1].hist(p_x, bins=100, alpha=0.5, color='orange')
axs[0, 1].set_title(r'$p_x$')
axs[1, 0].hist(p_y, bins=100, alpha=0.5, color='green')
axs[1, 0].set_title(r'$p_y$')
axs[1, 1].hist(p_z, bins=100, alpha=0.5, color='red')
axs[1, 1].set_title(r'$p_z$')

In [ ]:
from scipy.stats import skew
e_c_mirrored = np.concatenate([e_c, -e_c])
skew(np.array(e_c_mirrored)), skew(np.array(p_x)), skew(np.array(p_y)), skew(np.array(p_z))

The data is fairly normally distributed, so we scale it by the calculated normal from the Anderson tests

In [ ]:
from scipy.stats import anderson
and_ec = anderson(e_c_mirrored)
and_p_x = anderson(p_x)
and_p_y = anderson(p_y)
and_p_z = anderson(p_z)

and_ec, and_p_x, and_p_y, and_p_z

In [ ]:
scales = [an.fit_result.params.scale for an in [and_ec, and_p_x, and_p_y, and_p_z]]
final_scale = np.min(scales)
scales, final_scale

In [ ]:
e_c, p_x, p_y, p_z = (1/final_scale * arr for arr in [e_c, p_x, p_y, p_z])

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 10))
axs[0, 0].hist(e_c, bins=100, alpha=0.5, color='blue')
axs[0, 0].set_title(r'$E/c$')
axs[0, 1].hist(p_x, bins=100, alpha=0.5, color='orange')
axs[0, 1].set_title(r'$p_x$')
axs[1, 0].hist(p_y, bins=100, alpha=0.5, color='green')
axs[1, 0].set_title(r'$p_y$')
axs[1, 1].hist(p_z, bins=100, alpha=0.5, color='red')
axs[1, 1].set_title(r'$p_z$')

In [ ]:
anderson(e_c), anderson(p_x), anderson(p_y), anderson(p_z)